In [ ]:
!pip uninstall -y transformers
!pip install transformers==4.41.2
!pip uninstall -y peft

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 46.8 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.7.1
    Uninstalling huggingface_hub-1.7.1:
      Successfully uninstalled huggingface_hub-1.7.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
# from sklearn.model_selection

!pip install transformers datasets scikit-learn -q

import numpy as np
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/balanced_dataset_12k.csv')

In [ ]:
df.head()

,headline,label,confidence
0,"How JD(S) is quietly remaking its leadership team: As a veteran MLA is removed from core committee, ‘first family’ scions get key posts",sarcastic,0.65
1,"Waqf Bill: More than 12 hours of debate, and a washroom break that got Congress all wet",sarcastic,0.85
2,"India-U.S. interim trade deal: ‘Namaste Trump scored over Howdy Modi’, says Congress",sarcastic,0.75
3,"‘Richest’ CM Chandrababu Naidu’s draws almost entirely on one asset, Mamata records sharpest decline",sarcastic,0.45
4,Sunil Jakhar ‘resignation’: How ex-Congress leader struggled to find feet in BJP,sarcastic,0.45


In [ ]:
df = df.drop(columns=['confidence'], axis = 1)

In [ ]:
import re

def clean_text(text):
    text = str(text)

    # remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # remove HTML tags
    text = re.sub(r"<.*?>", "", text)

    # remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


df['text'] = df['headline'].apply(lambda x: clean_text(x))

In [ ]:
df.head()

,headline,label,text
0,"How JD(S) is quietly remaking its leadership team: As a veteran MLA is removed from core committee, ‘first family’ scions get key posts",sarcastic,"How JD(S) is quietly remaking its leadership team: As a veteran MLA is removed from core committee, ‘first family’ scions get key posts"
1,"Waqf Bill: More than 12 hours of debate, and a washroom break that got Congress all wet",sarcastic,"Waqf Bill: More than 12 hours of debate, and a washroom break that got Congress all wet"
2,"India-U.S. interim trade deal: ‘Namaste Trump scored over Howdy Modi’, says Congress",sarcastic,"India-U.S. interim trade deal: ‘Namaste Trump scored over Howdy Modi’, says Congress"
3,"‘Richest’ CM Chandrababu Naidu’s draws almost entirely on one asset, Mamata records sharpest decline",sarcastic,"‘Richest’ CM Chandrababu Naidu’s draws almost entirely on one asset, Mamata records sharpest decline"
4,Sunil Jakhar ‘resignation’: How ex-Congress leader struggled to find feet in BJP,sarcastic,Sunil Jakhar ‘resignation’: How ex-Congress leader struggled to find feet in BJP


In [ ]:
from huggingface_hub import login
login(token="###########################")

In [ ]:
texts = df["headline"].tolist()
labels = df["label"].tolist()

train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.1, stratify=labels, random_state=42
)

train_dataset = Dataset.from_dict({"text": train_texts, "label": train_labels})
val_dataset   = Dataset.from_dict({"text": val_texts, "label": val_labels})

In [ ]:
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128   # 🔥 optimized for speed
    )

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset   = val_dataset.map(tokenize, batched=True)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/10605 [00:00<?, ? examples/s]

Map:   0%|          | 0/1179 [00:00<?, ? examples/s]

In [ ]:
config = AutoConfig.from_pretrained(
    model_name,
    num_labels=2,
    hidden_dropout_prob=0.3,
    attention_probs_dropout_prob=0.3
)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    config=config
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Using:", device)

Using: cuda


In [ ]:
for param in model.bert.encoder.layer[:6].parameters():
    param.requires_grad = False

In [ ]:
training_args = TrainingArguments(
    output_dir="./results_major",

    # 🔥 GPU optimization
    per_device_train_batch_size=256,
    per_device_eval_batch_size=256,
    fp16=False,

    # training control
    learning_rate=2e-5,
    num_train_epochs=50,
    warmup_ratio=0.1,
    weight_decay=0.01,

    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,

    load_best_model_at_end=True,
    metric_for_best_model="f1",

    report_to="none"
)

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
def encode_labels(example):
    label_map = {
        "non-sarcastic": 0,
        "sarcastic": 1
    }
    example["label"] = label_map[example["label"]]
    return example

In [ ]:
train_dataset = train_dataset.map(encode_labels)
val_dataset = val_dataset.map(encode_labels)

Map:   0%|          | 0/10605 [00:00<?, ? examples/s]

KeyError: tensor(0)

In [ ]:
print(train_dataset[0])

{'label': tensor(0), 'input_ids': tensor([  101, 24954,  2097,  2272,  2000,  2373,  2065,  5152,  4494,  2024,
         4055,  1024,  9815, 24281,  2072,   102,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,  

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        # Overall
        "accuracy": accuracy_score(labels, preds),

        # Binary class metrics (positive = sarcastic)
        "precision": precision_score(labels, preds),
        "recall": recall_score(labels, preds),
        "f1": f1_score(labels, preds),

        # Macro (treat classes equally)
        "precision_macro": precision_score(labels, preds, average="macro"),
        "recall_macro": recall_score(labels, preds, average="macro"),
        "f1_macro": f1_score(labels, preds, average="macro"),

        # Weighted (accounts for imbalance)
        "precision_weighted": precision_score(labels, preds, average="weighted"),
        "recall_weighted": recall_score(labels, preds, average="weighted"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

# STEP 1 — FIX LABELS FIRST
# def encode_labels(example):
#     label_map = {
#         "non-sarcastic": 0,
#         "sarcastic": 1
#     }
#     example["label"] = label_map[example["label"]]
#     return example

# train_dataset = train_dataset.map(encode_labels)
# val_dataset = val_dataset.map(encode_labels)

# STEP 2 — REMOVE EXTRA COLUMNS (IMPORTANT)
train_dataset = train_dataset.remove_columns(
    [col for col in train_dataset.column_names if col not in ["input_ids", "attention_mask", "label"]]
)

val_dataset = val_dataset.remove_columns(
    [col for col in val_dataset.column_names if col not in ["input_ids", "attention_mask", "label"]]
)

# STEP 3 — SET FORMAT
train_dataset.set_format(type="torch")
val_dataset.set_format(type="torch")

# STEP 4 — NOW CREATE TRAINER (AFTER FIXES)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# STEP 5 — TRAIN
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Precision Macro,Recall Macro,F1 Macro,Precision Weighted,Recall Weighted,F1 Weighted
1,No log,0.654376,0.644614,0.787162,0.395586,0.526554,0.691996,0.644403,0.621050,0.691915,0.644614,0.621130
2,0.538100,0.663673,0.644614,0.801418,0.383701,0.518944,0.698368,0.644393,0.618584,0.698281,0.644614,0.618669
3,0.544800,0.656635,0.647159,0.799308,0.392190,0.526196,0.698530,0.646943,0.622557,0.698445,0.647159,0.622639
4,0.543400,0.610952,0.681934,0.784574,0.500849,0.611399,0.709224,0.681780,0.671098,0.709160,0.681934,0.671149
5,0.536500,0.712578,0.631043,0.805556,0.344652,0.482759,0.694579,0.630801,0.598004,0.694485,0.631043,0.598102
6,0.521700,0.659774,0.660729,0.805825,0.422750,0.554566,0.707510,0.660528,0.640297,0.707427,0.660729,0.640369
7,0.521700,0.625651,0.687023,0.786458,0.512733,0.620761,0.712726,0.686875,0.677167,0.712663,0.687023,0.677215
8,0.510200,0.636185,0.687871,0.785530,0.516129,0.622951,0.712841,0.687726,0.678335,0.712779,0.687871,0.678382
9,0.492400,0.657408,0.690416,0.790155,0.517827,0.625641,0.716011,0.690269,0.680861,0.715948,0.690416,0.680908
10,0.483600,0.721456,0.655640,0.806020,0.409168,0.542793,0.705283,0.655431,0.633301,0.705197,0.655640,0.633378


TrainOutput(global_step=2100, training_loss=0.39062558401198616, metrics={'train_runtime': 2866.2163, 'train_samples_per_second': 185.0, 'train_steps_per_second': 0.733, 'total_flos': 3.487865927616e+16, 'train_loss': 0.39062558401198616, 'epoch': 50.0})

In [ ]:
results = trainer.evaluate()
print(results)

{'eval_loss': 0.6581925749778748, 'eval_accuracy': 0.6912637828668363, 'eval_precision': 0.7892030848329049, 'eval_recall': 0.5212224108658744, 'eval_f1': 0.6278118609406953, 'eval_precision_macro': 0.7161205297582246, 'eval_recall_macro': 0.6911196800092083, 'eval_f1_macro': 0.6820218724993332, 'eval_precision_weighted': 0.716058542858246, 'eval_recall_weighted': 0.6912637828668363, 'eval_f1_weighted': 0.6820678521529028, 'eval_runtime': 2.4158, 'eval_samples_per_second': 488.046, 'eval_steps_per_second': 2.07, 'epoch': 50.0}


In [ ]:
trainer.save_model("/content/drive/MyDrive/Colab Notebooks/major projectr/sarcasm_bert_model")
tokenizer.save_pretrained("/content/drive/MyDrive/Colab Notebooks/major projectr/sarcasm_bert_model")

('/content/drive/MyDrive/Colab Notebooks/major projectr/sarcasm_bert_model/tokenizer_config.json',
 '/content/drive/MyDrive/Colab Notebooks/major projectr/sarcasm_bert_model/special_tokens_map.json',
 '/content/drive/MyDrive/Colab Notebooks/major projectr/sarcasm_bert_model/vocab.txt',
 '/content/drive/MyDrive/Colab Notebooks/major projectr/sarcasm_bert_model/added_tokens.json',
 '/content/drive/MyDrive/Colab Notebooks/major projectr/sarcasm_bert_model/tokenizer.json')

In [ ]:
from transformers import pipeline

clf1 = pipeline("text-classification", model="/content/drive/MyDrive/Colab Notebooks/major projectr/sarcasm_bert_model")

clf1("Government promises transparency, deletes all records")

[{'label': 'LABEL_0', 'score': 0.9165202975273132}]